In [0]:
# ============================================
# كود اكتشاف الملفات - نسخة مبسطة
# ============================================

from azure.storage.blob import BlobServiceClient

storage_account = "fraudanalytics"
access_key = "-------------------------------------"

connection_string = f"DefaultEndpointsProtocol=https;AccountName={storage_account};AccountKey={access_key};EndpointSuffix=core.windows.net"

print("=" * 60)
print("🔍 اكتشاف الـ Containers في Storage Account")
print("=" * 60)

try:
    blob_service_client = BlobServiceClient.from_connection_string(connection_string)
    
    # جلب جميع الـ Containers
    containers = list(blob_service_client.list_containers())
    
    if not containers:
        print("❌ لا يوجد أي Containers في هذا الحساب")
    else:
        print(f"✅ تم العثور على {len(containers)} Container(s):")
        for container in containers:
            print(f"\n📁 Container: {container.name}")
            
            # جلب الملفات في كل Container
            try:
                container_client = blob_service_client.get_container_client(container.name)
                blobs = list(container_client.list_blobs())
                
                if blobs:
                    print(f"   ✅ عدد الملفات: {len(blobs)}")
                    for blob in blobs[:10]:  # أول 10 ملفات بس
                        print(f"      📄 {blob.name} ({blob.size:,} bytes)")
                else:
                    print(f"   📭 Container فاضي")
            except Exception as e:
                print(f"   ⚠️ خطأ في قراءة الملفات: {e}")
                
except Exception as e:
    print(f"❌ فشل الاتصال: {e}")
    print("\nتأكد من:")
    print("  1. اسم الحساب صحيح: fraudanalytics")
    print("  2. Access Key صحيح")
    print("  3. الحساب موجود في Azure")

print("\n" + "=" * 60)

🔍 اكتشاف الـ Containers في Storage Account
✅ تم العثور على 3 Container(s):

📁 Container: bronze
   ✅ عدد الملفات: 4
      📄 telecom/landings/calls.csv (54,363,941 bytes)
      📄 telecom/landings/complaints.csv (70,005 bytes)
      📄 telecom/landings/customers.csv (476,892 bytes)
      📄 telecom/landings/recharges.csv (994,946 bytes)

📁 Container: gold
   📭 Container فاضي

📁 Container: silver
   📭 Container فاضي



In [0]:
# ============================================
# Fraud Analytics Pipeline - الإصدار النهائي
# (بدون أي أخطاء - مبني على الأعمدة الفعلية)
# ============================================

from azure.storage.blob import BlobServiceClient
import pandas as pd
import io
import re
from pyspark.sql.functions import col, trim, upper, when, lit, to_date, year, month, hour
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

# تثبيت pyarrow
%pip install pyarrow -q

# ============================================
# 1. إعدادات الاتصال
# ============================================

storage_account = "fraudanalytics"
container_bronze = "bronze"
container_silver = "silver"
folder_path = "telecom/landings/"
access_key = "CGix739q4kl4z2rWWrv5lvOiK0GDDB+LEyjfkabsgceOkcNcXx+PMpZVa9oe5fU8t9U82g78hUx++AStPkvhTw=="

connection_string = f"DefaultEndpointsProtocol=https;AccountName={storage_account};AccountKey={access_key};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_bronze)
silver_container_client = blob_service_client.get_container_client(container_silver)

print("=" * 70)
print("🚀 FRAUD ANALYTICS PIPELINE")
print("=" * 70)
print(f"✅ Storage Account: {storage_account}")
print(f"✅ Source Container: {container_bronze}")
print(f"✅ Target Container: {container_silver} (Parquet)")

# ============================================
# 2. دالة تحميل الملفات
# ============================================

def load_csv(blob_name):
    full_path = folder_path + blob_name
    blob_client = container_client.get_blob_client(full_path)
    blob_data = blob_client.download_blob().readall()
    df_pandas = pd.read_csv(io.BytesIO(blob_data), dtype=str)
    print(f"   ✅ {blob_name} ({len(df_pandas):,} صف)")
    return spark.createDataFrame(df_pandas)

# ============================================
# 3. تحميل الملفات
# ============================================

print("\n📖 جاري تحميل الملفات...")

df_customers = load_csv("01_customers_raw.csv")
df_calls = load_csv("02_calls_raw.csv")
df_transactions = load_csv("03_transactions_raw.csv")
df_complaints = load_csv("04_complaints_raw.csv")

print(f"\n   customers   : {df_customers.count():,} صف")
print(f"   calls       : {df_calls.count():,} صف")
print(f"   transactions: {df_transactions.count():,} صف")
print(f"   complaints  : {df_complaints.count():,} صف")

# ============================================
# 4. دالة آمنة لتحويل التاريخ (لجميع التنسيقات)
# ============================================

def safe_parse_date(date_str):
    if date_str is None or pd.isna(date_str) or str(date_str).strip() == "":
        return None
    date_str = str(date_str).strip()
    # التنسيق: M/d/yyyy (مثل 3/25/2026)
    pattern1 = r'^\d{1,2}/\d{1,2}/\d{4}$'
    # التنسيق: yyyy-MM-dd (مثل 2026-03-14)
    pattern2 = r'^\d{4}-\d{2}-\d{2}$'
    
    if re.match(pattern1, date_str):
        try:
            parts = date_str.split('/')
            return f"{parts[2]}-{parts[0].zfill(2)}-{parts[1].zfill(2)}"
        except:
            return None
    elif re.match(pattern2, date_str):
        return date_str
    return None

safe_parse_date_udf = udf(safe_parse_date, StringType())

# ============================================
# 5. دالة آمنة لتحويل الطابع الزمني
# ============================================

def safe_parse_timestamp(ts_str):
    if ts_str is None or pd.isna(ts_str) or str(ts_str).strip() == "":
        return None
    ts_str = str(ts_str).strip()
    # التنسيق: yyyy-MM-dd HH:mm:ss
    pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'
    if re.match(pattern, ts_str):
        return ts_str
    return None

safe_parse_timestamp_udf = udf(safe_parse_timestamp, StringType())

# ============================================
# 6. تنظيف العملاء
# ============================================

print("\n🧹 جاري تنظيف البيانات...")

df_customers_clean = df_customers \
    .dropDuplicates(["customer_id"]) \
    .filter(col("customer_id").isNotNull()) \
    .fillna({
        "full_name": "UNKNOWN",
        "phone_number": "0000000000",
        "age": "0",
        "district": "UNKNOWN",
        "subscription_plan": "UNKNOWN",
        "credit_score": "0.0",
        "is_fraud": "0"
    }) \
    .withColumn("full_name", upper(trim(col("full_name")))) \
    .withColumn("district", upper(trim(col("district")))) \
    .withColumn("subscription_plan", upper(trim(col("subscription_plan")))) \
    .withColumn("age", col("age").cast(DoubleType()).cast(IntegerType())) \
    .withColumn("age", when(col("age") < 0, 0).otherwise(col("age"))) \
    .withColumn("credit_score", col("credit_score").cast(DoubleType())) \
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType())) \
    .withColumn("registration_date_temp", safe_parse_date_udf(col("registration_date"))) \
    .withColumn("registration_date", to_date(col("registration_date_temp"), "yyyy-MM-dd")) \
    .drop("registration_date_temp")

print("   ✅ customers")

# إحصائيات العملاء
print(f"      👥 عدد العملاء: {df_customers_clean.count():,}")
avg_age = df_customers_clean.select("age").agg({"age": "avg"}).collect()[0][0] or 0
print(f"      📊 متوسط العمر: {avg_age:.1f}")
fraud_customers = df_customers_clean.filter(col("is_fraud") == 1).count()
print(f"      ⚠️ عملاء احتيال: {fraud_customers:,}")

# ============================================
# 7. تنظيف المكالمات (باستخدام الأعمدة الموجودة فقط)
# ============================================

df_calls_clean = df_calls \
    .dropDuplicates(["call_id"]) \
    .filter(col("call_id").isNotNull()) \
    .fillna({
        "duration_sec": "0",
        "destination_net": "UNKNOWN",
        "call_type": "UNKNOWN",
        "cost_egp": "0.0",
        "is_fraud_call": "0"
    }) \
    .withColumn("duration_sec", col("duration_sec").cast(IntegerType())) \
    .withColumn("duration_sec", when(col("duration_sec") < 0, 0).otherwise(col("duration_sec"))) \
    .withColumn("cost_egp", col("cost_egp").cast(DoubleType())) \
    .withColumn("cost_egp", when(col("cost_egp") < 0, 0.0).otherwise(col("cost_egp"))) \
    .withColumn("is_fraud_call", col("is_fraud_call").cast(IntegerType())) \
    .withColumn("destination_net", upper(trim(col("destination_net")))) \
    .withColumn("call_type", upper(trim(col("call_type")))) \
    .withColumn("timestamp_temp", safe_parse_timestamp_udf(col("timestamp"))) \
    .withColumn("call_date", to_date(col("timestamp_temp"), "yyyy-MM-dd")) \
    .withColumn("call_hour", hour(col("timestamp_temp"))) \
    .drop("timestamp_temp")

print("   ✅ calls")

# إحصائيات المكالمات
total_calls = df_calls_clean.count()
fraud_calls = df_calls_clean.filter(col("is_fraud_call") == 1).count()
fraud_percent = (fraud_calls / total_calls * 100) if total_calls > 0 else 0
print(f"      📞 إجمالي المكالمات: {total_calls:,}")
print(f"      🔴 مكالمات احتيالية: {fraud_calls:,} ({fraud_percent:.2f}%)")

# ============================================
# 8. تنظيف المعاملات
# ============================================

df_transactions_clean = df_transactions \
    .dropDuplicates(["transaction_id"]) \
    .filter(col("transaction_id").isNotNull()) \
    .fillna({
        "transaction_type": "UNKNOWN",
        "amount_egp": "0.0",
        "channel": "UNKNOWN",
        "status": "UNKNOWN",
        "is_fraud": "0"
    }) \
    .withColumn("amount_egp", col("amount_egp").cast(DoubleType())) \
    .withColumn("amount_egp", when(col("amount_egp") < 0, 0.0).otherwise(col("amount_egp"))) \
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType())) \
    .withColumn("transaction_type", upper(trim(col("transaction_type")))) \
    .withColumn("channel", upper(trim(col("channel")))) \
    .withColumn("status", upper(trim(col("status")))) \
    .withColumn("date_temp", safe_parse_date_udf(col("date"))) \
    .withColumn("date", to_date(col("date_temp"), "yyyy-MM-dd")) \
    .withColumn("transaction_year", year(col("date"))) \
    .withColumn("transaction_month", month(col("date"))) \
    .drop("date_temp")

print("   ✅ transactions")

# إحصائيات المعاملات
total_trans = df_transactions_clean.count()
fraud_trans = df_transactions_clean.filter(col("is_fraud") == 1).count()
fraud_trans_percent = (fraud_trans / total_trans * 100) if total_trans > 0 else 0
total_amount = df_transactions_clean.agg({"amount_egp": "sum"}).collect()[0][0] or 0
print(f"      💰 إجمالي المعاملات: {total_trans:,}")
print(f"      🔴 معاملات احتيالية: {fraud_trans:,} ({fraud_trans_percent:.2f}%)")
print(f"      💵 إجمالي المبالغ: {total_amount:,.2f} جنيه")

# ============================================
# 9. تنظيف الشكاوى
# ============================================

df_complaints_clean = df_complaints \
    .dropDuplicates(["complaint_id"]) \
    .filter(col("complaint_id").isNotNull()) \
    .fillna({
        "complaint_type": "OTHER",
        "severity": "MEDIUM",
        "resolution": "PENDING",
        "days_to_resolve": "0"
    }) \
    .withColumn("complaint_type", upper(trim(col("complaint_type")))) \
    .withColumn("severity", upper(trim(col("severity")))) \
    .withColumn("resolution", upper(trim(col("resolution")))) \
    .withColumn("days_to_resolve", col("days_to_resolve").cast(IntegerType())) \
    .withColumn("days_to_resolve", when(col("days_to_resolve") < 0, 0).otherwise(col("days_to_resolve"))) \
    .withColumn("date_temp", safe_parse_date_udf(col("date"))) \
    .withColumn("date", to_date(col("date_temp"), "yyyy-MM-dd")) \
    .drop("date_temp")

print("   ✅ complaints")

# إحصائيات الشكاوى
total_comp = df_complaints_clean.count()
high_severity = df_complaints_clean.filter(col("severity") == "HIGH").count()
med_severity = df_complaints_clean.filter(col("severity") == "MEDIUM").count()
low_severity = df_complaints_clean.filter(col("severity") == "LOW").count()
print(f"      📋 إجمالي الشكاوى: {total_comp:,}")
print(f"      🔴 شكاوى عالية: {high_severity:,}")
print(f"      🟠 شكاوى متوسطة: {med_severity:,}")
print(f"      🟢 شكاوى منخفضة: {low_severity:,}")

# ============================================
# 10. عرض عينة بعد التنظيف
# ============================================

print("\n📋 عينة من العملاء (بعد التنظيف):")
df_customers_clean.select("customer_id", "full_name", "age", "district", "subscription_plan").show(5, truncate=False)

print("\n📋 عينة من المكالمات (بعد التنظيف):")
df_calls_clean.select("call_id", "customer_id", "duration_sec", "call_type", "is_fraud_call").show(5, truncate=False)

print("\n📋 عينة من المعاملات (بعد التنظيف):")
df_transactions_clean.select("transaction_id", "customer_id", "amount_egp", "status", "is_fraud").show(5, truncate=False)

# ============================================
# 11. تحليلات سريعة
# ============================================

print("\n" + "=" * 70)
print("📊 FRAUD INSIGHTS")
print("=" * 70)

# العملاء المتورطين في احتيال
fraud_call_customers = df_calls_clean.filter(col("is_fraud_call") == 1).select("customer_id").distinct().count()
fraud_trans_customers = df_transactions_clean.filter(col("is_fraud") == 1).select("customer_id").distinct().count()
print(f"\n👥 عملاء بمكالمات احتيالية: {fraud_call_customers:,}")
print(f"👥 عملاء بمعاملات احتيالية: {fraud_trans_customers:,}")

# توزيع المكالمات حسب النوع
print("\n📞 توزيع المكالمات حسب النوع:")
df_calls_clean.groupBy("call_type").count().orderBy(col("count").desc()).show()

# توزيع المعاملات حسب الحالة
print("\n💰 توزيع المعاملات حسب الحالة:")
df_transactions_clean.groupBy("status").count().orderBy(col("count").desc()).show()

# أعلى 10 عملاء في عدد المكالمات
print("\n📞 أعلى 10 عملاء في عدد المكالمات:")
df_calls_clean.groupBy("customer_id").count() \
    .orderBy(col("count").desc()) \
    .limit(10) \
    .show()

# ============================================
# 12. حفظ البيانات كـ Parquet
# ============================================

def save_parquet(df, blob_name, container_client):
    try:
        pandas_df = df.toPandas()
        buffer = io.BytesIO()
        pandas_df.to_parquet(buffer, index=False, engine='pyarrow')
        buffer.seek(0)
        blob_client = container_client.get_blob_client(blob_name)
        blob_client.upload_blob(buffer.getvalue(), overwrite=True)
        print(f"   ✅ {blob_name} ({len(pandas_df):,} صف, {buffer.getbuffer().nbytes:,} bytes)")
    except Exception as e:
        print(f"   ⚠️ خطأ في حفظ {blob_name}: {str(e)[:100]}")

print(f"\n💾 جاري حفظ البيانات النظيفة كـ PARQUET...")

# التأكد من وجود Container silver
try:
    silver_container_client.get_container_properties()
    print(f"   ✅ Container '{container_silver}' موجود")
except:
    try:
        blob_service_client.create_container(container_silver)
        print(f"   ✅ تم إنشاء Container '{container_silver}'")
    except:
        print(f"   ⚠️ Container '{container_silver}' قد يكون موجوداً بالفعل")

save_parquet(df_customers_clean, "01_customers_cleaned.parquet", silver_container_client)
save_parquet(df_calls_clean, "02_calls_cleaned.parquet", silver_container_client)
save_parquet(df_transactions_clean, "03_transactions_cleaned.parquet", silver_container_client)
save_parquet(df_complaints_clean, "04_complaints_cleaned.parquet", silver_container_client)

print(f"\n✅ تم الحفظ بنجاح")

# ============================================
# 13. الملخص النهائي
# ============================================

print("\n" + "=" * 70)
print("📊 FINAL SUMMARY")
print("=" * 70)
print(f"  customers   : {df_customers_clean.count():>10,} سجل")
print(f"  calls       : {df_calls_clean.count():>10,} سجل")
print(f"  transactions: {df_transactions_clean.count():>10,} سجل")
print(f"  complaints  : {df_complaints_clean.count():>10,} سجل")
print("=" * 70)

print("\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print(f"\n📁 البيانات النظيفة في Container '{container_silver}'")
print("\n📖 أسماء الملفات:")
print("   - 01_customers_cleaned.parquet")
print("   - 02_calls_cleaned.parquet")
print("   - 03_transactions_cleaned.parquet")
print("   - 04_complaints_cleaned.parquet")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
🚀 FRAUD ANALYTICS PIPELINE
✅ Storage Account: fraudanalytics
✅ Source Container: bronze
✅ Target Container: silver (Parquet)

📖 جاري تحميل الملفات...
   ✅ 01_customers_raw.csv (5,075 صف)
   ✅ 02_calls_raw.csv (1,439,508 صف)
   ✅ 03_transactions_raw.csv (61,979 صف)
   ✅ 04_complaints_raw.csv (600 صف)

   customers   : 5,075 صف
   calls       : 1,439,508 صف
   transactions: 61,979 صف
   complaints  : 600 صف

🧹 جاري تنظيف البيانات...
   ✅ customers
      👥 عدد العملاء: 5,000
      📊 متوسط العمر: 48.4
      ⚠️ عملاء احتيال: 150
   ✅ calls
      📞 إجمالي المكالمات: 1,432,346
      🔴 مكالمات احتيالية: 123,255 (8.61%)
   ✅ transactions
      💰 إجمالي المعاملات: 61,365
      🔴 معاملات احتيالية: 5,744 (9.36%)
      💵 إجمالي المبالغ: 42,784,593.13 جنيه
   ✅ complaints
      📋 إجمالي الشكاوى: 600
      🔴 شكاوى عالية: 135
      🟠 شكاوى متوسطة: 136
      🟢 شكاوى منخفضة: 109

📋 

In [0]:
# ============================================
# Fraud Analytics Pipeline - الإصدار النهائي المضمون
# (بدون أي أخطاء - يعمل على Serverless)
# ============================================

from azure.storage.blob import BlobServiceClient
import pandas as pd
import io
import re
from pyspark.sql.functions import col, trim, upper, when, lit, to_date, year, month
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

# ============================================
# 1. إعدادات الاتصال
# ============================================

storage_account = "fraudanalytics"
container_bronze = "bronze"
container_silver = "silver"
folder_path = "telecom/landings/"
access_key = "CGix739q4kl4z2rWWrv5lvOiK0GDDB+LEyjfkabsgceOkcNcXx+PMpZVa9oe5fU8t9U82g78hUx++AStPkvhTw=="

connection_string = f"DefaultEndpointsProtocol=https;AccountName={storage_account};AccountKey={access_key};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(container_bronze)
silver_container_client = blob_service_client.get_container_client(container_silver)

print("=" * 70)
print("🚀 FRAUD ANALYTICS PIPELINE")
print("=" * 70)
print(f"✅ Storage Account: {storage_account}")
print(f"✅ Source Container: {container_bronze}")
print(f"✅ Target Container: {container_silver}")

# ============================================
# 2. دالة تحميل الملفات
# ============================================

def load_csv(blob_name):
    full_path = folder_path + blob_name
    blob_client = container_client.get_blob_client(full_path)
    blob_data = blob_client.download_blob().readall()
    df_pandas = pd.read_csv(io.BytesIO(blob_data), dtype=str)
    print(f"   ✅ {blob_name} ({len(df_pandas):,} صف)")
    return spark.createDataFrame(df_pandas)

# ============================================
# 3. دالة حفظ DataFrame
# ============================================

def save_df_to_blob(df, blob_name, container_client):
    """حفظ DataFrame كـ CSV في Blob Storage"""
    pandas_df = df.toPandas()
    csv_buffer = io.StringIO()
    pandas_df.to_csv(csv_buffer, index=False)
    csv_buffer.seek(0)
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)
    print(f"   ✅ {blob_name} ({len(pandas_df):,} صف)")

# ============================================
# 4. دالة آمنة للتواريخ (بدون timestamp)
# ============================================

def safe_parse_date(date_str):
    if date_str is None or pd.isna(date_str) or str(date_str).strip() == "":
        return None
    date_str = str(date_str).strip()
    # صيغة M/d/yyyy
    pattern1 = r'^\d{1,2}/\d{1,2}/\d{4}$'
    # صيغة yyyy-MM-dd
    pattern2 = r'^\d{4}-\d{2}-\d{2}$'
    
    if re.match(pattern1, date_str):
        try:
            parts = date_str.split('/')
            return f"{parts[2]}-{parts[0].zfill(2)}-{parts[1].zfill(2)}"
        except:
            return None
    elif re.match(pattern2, date_str):
        return date_str
    return None

safe_parse_date_udf = udf(safe_parse_date, StringType())

# ============================================
# 5. تحميل الملفات
# ============================================

print("\n📖 جاري تحميل الملفات...")

df_customers = load_csv("01_customers_raw.csv")
df_calls = load_csv("02_calls_raw.csv")
df_transactions = load_csv("03_transactions_raw.csv")
df_complaints = load_csv("04_complaints_raw.csv")

print(f"\n   customers   : {df_customers.count():,} صف")
print(f"   calls       : {df_calls.count():,} صف")
print(f"   transactions: {df_transactions.count():,} صف")
print(f"   complaints  : {df_complaints.count():,} صف")

# ============================================
# 6. تنظيف العملاء
# ============================================

print("\n🧹 جاري تنظيف البيانات...")

df_customers_clean = df_customers \
    .dropDuplicates(["customer_id"]) \
    .filter(col("customer_id").isNotNull()) \
    .fillna({
        "full_name": "UNKNOWN",
        "phone_number": "0000000000",
        "age": "0",
        "district": "UNKNOWN",
        "subscription_plan": "UNKNOWN",
        "credit_score": "0.0",
        "is_fraud": "0"
    }) \
    .withColumn("full_name", upper(trim(col("full_name")))) \
    .withColumn("district", upper(trim(col("district")))) \
    .withColumn("subscription_plan", upper(trim(col("subscription_plan")))) \
    .withColumn("age", col("age").cast(DoubleType()).cast(IntegerType())) \
    .withColumn("age", when(col("age") < 0, 0).otherwise(col("age"))) \
    .withColumn("credit_score", col("credit_score").cast(DoubleType())) \
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType())) \
    .withColumn("registration_date_temp", safe_parse_date_udf(col("registration_date"))) \
    .withColumn("registration_date", to_date(col("registration_date_temp"), "yyyy-MM-dd")) \
    .drop("registration_date_temp")

print("   ✅ customers")

# ============================================
# 7. تنظيف المكالمات (بدون تحويل timestamp معقد)
# ============================================

df_calls_clean = df_calls \
    .dropDuplicates(["call_id"]) \
    .filter(col("call_id").isNotNull()) \
    .fillna({
        "duration_sec": "0",
        "destination_net": "UNKNOWN",
        "call_type": "UNKNOWN",
        "cost_egp": "0.0",
        "is_fraud_call": "0"
    }) \
    .withColumn("duration_sec", col("duration_sec").cast(IntegerType())) \
    .withColumn("duration_sec", when(col("duration_sec") < 0, 0).otherwise(col("duration_sec"))) \
    .withColumn("cost_egp", col("cost_egp").cast(DoubleType())) \
    .withColumn("cost_egp", when(col("cost_egp") < 0, 0.0).otherwise(col("cost_egp"))) \
    .withColumn("is_fraud_call", col("is_fraud_call").cast(IntegerType())) \
    .withColumn("destination_net", upper(trim(col("destination_net")))) \
    .withColumn("call_type", upper(trim(col("call_type")))) \
    .withColumn("call_date_str", regexp_extract(col("timestamp"), r'^(\d{4}-\d{2}-\d{2})', 1)) \
    .withColumn("call_date", to_date(col("call_date_str"), "yyyy-MM-dd")) \
    .drop("call_date_str")

print("   ✅ calls")

# ============================================
# 8. تنظيف المعاملات
# ============================================

df_transactions_clean = df_transactions \
    .dropDuplicates(["transaction_id"]) \
    .filter(col("transaction_id").isNotNull()) \
    .fillna({
        "transaction_type": "UNKNOWN",
        "amount_egp": "0.0",
        "channel": "UNKNOWN",
        "status": "UNKNOWN",
        "is_fraud": "0"
    }) \
    .withColumn("amount_egp", col("amount_egp").cast(DoubleType())) \
    .withColumn("amount_egp", when(col("amount_egp") < 0, 0.0).otherwise(col("amount_egp"))) \
    .withColumn("is_fraud", col("is_fraud").cast(IntegerType())) \
    .withColumn("transaction_type", upper(trim(col("transaction_type")))) \
    .withColumn("channel", upper(trim(col("channel")))) \
    .withColumn("status", upper(trim(col("status")))) \
    .withColumn("date_temp", safe_parse_date_udf(col("date"))) \
    .withColumn("date", to_date(col("date_temp"), "yyyy-MM-dd")) \
    .withColumn("transaction_year", year(col("date"))) \
    .withColumn("transaction_month", month(col("date"))) \
    .drop("date_temp")

print("   ✅ transactions")

# ============================================
# 9. تنظيف الشكاوى
# ============================================

df_complaints_clean = df_complaints \
    .dropDuplicates(["complaint_id"]) \
    .filter(col("complaint_id").isNotNull()) \
    .fillna({
        "complaint_type": "OTHER",
        "severity": "MEDIUM",
        "resolution": "PENDING",
        "days_to_resolve": "0"
    }) \
    .withColumn("complaint_type", upper(trim(col("complaint_type")))) \
    .withColumn("severity", upper(trim(col("severity")))) \
    .withColumn("resolution", upper(trim(col("resolution")))) \
    .withColumn("days_to_resolve", col("days_to_resolve").cast(DoubleType()).cast(IntegerType())) \
    .withColumn("days_to_resolve", when(col("days_to_resolve") < 0, 0).otherwise(col("days_to_resolve"))) \
    .withColumn("date_temp", safe_parse_date_udf(col("date"))) \
    .withColumn("date", to_date(col("date_temp"), "yyyy-MM-dd")) \
    .drop("date_temp")

print("   ✅ complaints")

print("\n✅ تم تنظيف جميع الملفات")

# ============================================
# 10. إحصائيات سريعة
# ============================================

print("\n" + "=" * 70)
print("📊 إحصائيات سريعة")
print("=" * 70)

print(f"\n👥 العملاء: {df_customers_clean.count():,}")
print(f"   - عملاء احتيال: {df_customers_clean.filter(col('is_fraud') == 1).count():,}")

print(f"\n📞 المكالمات: {df_calls_clean.count():,}")
print(f"   - مكالمات احتيالية: {df_calls_clean.filter(col('is_fraud_call') == 1).count():,}")

print(f"\n💰 المعاملات: {df_transactions_clean.count():,}")
print(f"   - معاملات احتيالية: {df_transactions_clean.filter(col('is_fraud') == 1).count():,}")
total_amount = df_transactions_clean.agg({"amount_egp": "sum"}).collect()[0][0] or 0
print(f"   - إجمالي المبالغ: {total_amount:,.2f} جنيه")

print(f"\n📋 الشكاوى: {df_complaints_clean.count():,}")
print(f"   - شكاوى عالية: {df_complaints_clean.filter(col('severity') == 'HIGH').count():,}")

# ============================================
# 11. حفظ البيانات
# ============================================

print("\n" + "=" * 70)
print("💾 جاري حفظ البيانات النظيفة")
print("=" * 70)

# التأكد من وجود Container silver
try:
    silver_container_client.get_container_properties()
    print(f"   ✅ Container '{container_silver}' موجود")
except:
    blob_service_client.create_container(container_silver)
    print(f"   ✅ تم إنشاء Container '{container_silver}'")

# حفظ الملفات
save_df_to_blob(df_customers_clean, "01_customers_cleaned.csv", silver_container_client)
save_df_to_blob(df_calls_clean, "02_calls_cleaned.csv", silver_container_client)
save_df_to_blob(df_transactions_clean, "03_transactions_cleaned.csv", silver_container_client)
save_df_to_blob(df_complaints_clean, "04_complaints_cleaned.csv", silver_container_client)

# ============================================
# 12. التحقق من الملفات
# ============================================

print("\n📁 الملفات المحفوظة في Container 'silver':")
try:
    blobs = list(silver_container_client.list_blobs())
    if blobs:
        for blob in blobs:
            print(f"   📄 {blob.name} ({blob.size:,} bytes)")
    else:
        print("   ⚠️ لا توجد ملفات")
except Exception as e:
    print(f"   ⚠️ خطأ في العرض: {e}")

# ============================================
# 13. الملخص النهائي
# ============================================

print("\n" + "=" * 70)
print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 70)
print(f"\n📁 البيانات النظيفة موجودة في:")
print(f"   Storage Account: {storage_account}")
print(f"   Container: {container_silver}")
print(f"\n📖 أسماء الملفات:")
print(f"   - 01_customers_cleaned.csv")
print(f"   - 02_calls_cleaned.csv")
print(f"   - 03_transactions_cleaned.csv")
print(f"   - 04_complaints_cleaned.csv")

🚀 FRAUD ANALYTICS PIPELINE
✅ Storage Account: fraudanalytics
✅ Source Container: bronze
✅ Target Container: silver

📖 جاري تحميل الملفات...
   ✅ 01_customers_raw.csv (5,075 صف)
   ✅ 02_calls_raw.csv (1,439,508 صف)
   ✅ 03_transactions_raw.csv (61,979 صف)
   ✅ 04_complaints_raw.csv (600 صف)

   customers   : 5,075 صف
   calls       : 1,439,508 صف
   transactions: 61,979 صف
   complaints  : 600 صف

🧹 جاري تنظيف البيانات...
   ✅ customers
   ✅ calls
   ✅ transactions
   ✅ complaints

✅ تم تنظيف جميع الملفات

📊 إحصائيات سريعة

👥 العملاء: 5,000
   - عملاء احتيال: 150

📞 المكالمات: 1,432,346
   - مكالمات احتيالية: 123,255

💰 المعاملات: 61,365
   - معاملات احتيالية: 5,744
   - إجمالي المبالغ: 42,784,593.13 جنيه

📋 الشكاوى: 600
   - شكاوى عالية: 135

💾 جاري حفظ البيانات النظيفة
   ✅ Container 'silver' موجود
   ✅ 01_customers_cleaned.csv (5,000 صف)
   ✅ 02_calls_cleaned.csv (1,432,346 صف)
   ✅ 03_transactions_cleaned.csv (61,365 صف)
   ✅ 04_complaints_cleaned.csv (600 صف)

📁 الملفات المحفوظة في